In [ ]:
!pip install streamlit

In [ ]:
import streamlit as st

print("Streamlit Version:", st.__version__)
print("Streamlit Installed Successfully!")

Streamlit Version: 1.67.0
Streamlit Installed Successfully!


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

project_path = "/content/drive/MyDrive/SmartFood_Rescue"

model_path = project_path + "/models"

print("Project Path:", project_path)
print("Model Path:", model_path)

Project Path: /content/drive/MyDrive/SmartFood_Rescue
Model Path: /content/drive/MyDrive/SmartFood_Rescue/models


In [ ]:
required_files = [
    "ann_regressor.pth",
    "ann_classifier.pth",
    "preprocessor.pkl",
    "demand_thresholds.pkl"
]

print("Checking Saved Model Files:\n")

for file in required_files:

    file_path = os.path.join(model_path, file)

    print(file, ":", os.path.exists(file_path))

Checking Saved Model Files:

ann_regressor.pth : True
ann_classifier.pth : True
preprocessor.pkl : True
demand_thresholds.pkl : True


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
import os

# Device Configuration
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# Load Preprocessor
preprocessor = joblib.load(
    model_path + "/preprocessor.pkl"
)

# Load Demand Thresholds
thresholds = joblib.load(
    model_path + "/demand_thresholds.pkl"
)

low_threshold = thresholds["low_threshold"]
high_threshold = thresholds["high_threshold"]

class_mapping = thresholds["class_mapping"]

print("\nPreprocessor Loaded Successfully!")

print("Low Threshold:", low_threshold)
print("High Threshold:", high_threshold)
print("Class Mapping:", class_mapping)

Device: cpu

Preprocessor Loaded Successfully!
Low Threshold: 80.0
High Threshold: 243.0
Class Mapping: {0: 'Low', 1: 'Medium', 2: 'High'}


In [ ]:
# Load Prepared Training Dataset
train_data = pd.read_csv(
    "/content/drive/MyDrive/food_dataset/train_prepared.csv"
)

# Select Input Features
X_sample = train_data.drop(
    columns=["id", "num_orders"]
)

# Transform Features
X_sample_processed = preprocessor.transform(X_sample)

# Convert Sparse Matrix to Dense
if hasattr(X_sample_processed, "toarray"):
    X_sample_processed = X_sample_processed.toarray()

# Determine Input Features
input_features = X_sample_processed.shape[1]

print("Input Features:", input_features)
print("Preprocessing Verified Successfully!")

Input Features: 217
Preprocessing Verified Successfully!


In [ ]:
class ANNRegressor(nn.Module):

    def __init__(self):
        super(ANNRegressor, self).__init__()

        self.model = nn.Sequential(

            nn.Linear(input_features, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x)


# Initialize Model
ann_regressor = ANNRegressor().to(device)

# Load Saved Weights
ann_regressor.load_state_dict(
    torch.load(
        model_path + "/ann_regressor.pth",
        map_location=device,
        weights_only=True
    )
)

ann_regressor.eval()

print("ANN Regression Model Loaded Successfully!")
print(ann_regressor)

ANN Regression Model Loaded Successfully!
ANNRegressor(
  (model): Sequential(
    (0): Linear(in_features=217, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [ ]:
class ANNClassifier(nn.Module):

    def __init__(self):
        super(ANNClassifier, self).__init__()

        self.model = nn.Sequential(

            nn.Linear(input_features, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 3)
        )

    def forward(self, x):
        return self.model(x)


# Initialize Model
ann_classifier = ANNClassifier().to(device)

# Load Saved Weights
ann_classifier.load_state_dict(
    torch.load(
        model_path + "/ann_classifier.pth",
        map_location=device,
        weights_only=True
    )
)

ann_classifier.eval()

print("ANN Classification Model Loaded Successfully!")
print(ann_classifier)

ANN Classification Model Loaded Successfully!
ANNClassifier(
  (model): Sequential(
    (0): Linear(in_features=217, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=3, bias=True)
  )
)


In [ ]:
print("SMARTFOOD RESCUE MODEL STATUS")

print("\nRegression Model:",
      "Loaded" if ann_regressor is not None else "Not Loaded")

print("Classification Model:",
      "Loaded" if ann_classifier is not None else "Not Loaded")

print("Preprocessor:",
      "Loaded" if preprocessor is not None else "Not Loaded")

print("Demand Thresholds:",
      "Loaded" if thresholds is not None else "Not Loaded")

print("\nDevice:", device)
print("Input Features:", input_features)

SMARTFOOD RESCUE MODEL STATUS

Regression Model: Loaded
Classification Model: Loaded
Preprocessor: Loaded
Demand Thresholds: Loaded

Device: cpu
Input Features: 217


In [ ]:
# Load Prepared Training Data

train_data = pd.read_csv(
    "/content/drive/MyDrive/food_dataset/train_prepared.csv"
)

# Identify Model Input Features
feature_columns = train_data.drop(
    columns=["id", "num_orders"]
).columns.tolist()

# Extract Unique Categorical Values
categorical_columns = [
    "category",
    "cuisine",
    "center_type",
    "center_id",
    "meal_id",
    "city_code",
    "region_code"
]

print("Training Data Loaded Successfully!")
print("Total Records:", len(train_data))
print("Total Input Columns:", len(feature_columns))

print("\nAvailable Categories:")
for column in categorical_columns:
    print(f"{column}: {train_data[column].nunique()} unique values")

print("\nFeature Columns:")
print(feature_columns)

Training Data Loaded Successfully!
Total Records: 456548
Total Input Columns: 16

Available Categories:
category: 14 unique values
cuisine: 4 unique values
center_type: 3 unique values
center_id: 77 unique values
meal_id: 51 unique values
city_code: 51 unique values
region_code: 8 unique values

Feature Columns:
['week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured', 'category', 'cuisine', 'city_code', 'region_code', 'center_type', 'op_area', 'discount_amount', 'discount_percentage', 'promotion_active']


In [ ]:
%%writefile /content/drive/MyDrive/SmartFood_Rescue/app.py

import streamlit as st

st.set_page_config(
    page_title="SmartFood Rescue",
    page_icon="🍲",
    layout="wide"
)

st.title("🍲 SmartFood Rescue")
st.subheader("Food Demand Forecasting and Classification")

st.write(
    """
    SmartFood Rescue uses Artificial Neural Networks (ANN)
    to forecast food demand and help reduce food wastage.
    """
)

st.info("Welcome! Your Streamlit application is being developed.")

st.markdown("---")

st.markdown(
    """
    ### Project Features

    - ANN Regression – Predict expected food orders.
    - ANN Classification – Classify demand as Low, Medium, or High.
    - Surplus Estimation – Estimate potential surplus using prepared quantity.
    - Food Rescue Recommendations – Suggest ways to manage potential surplus.
    """
)

Writing /content/drive/MyDrive/SmartFood_Rescue/app.py


In [ ]:
%%writefile -a /content/drive/MyDrive/SmartFood_Rescue/app.py

import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Project Paths
PROJECT_PATH = "/content/drive/MyDrive/SmartFood_Rescue"
MODEL_PATH = os.path.join(PROJECT_PATH, "models")
DATASET_PATH = "/content/drive/MyDrive/food_dataset"

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ANN Regression Architecture
class ANNRegressor(nn.Module):
    def __init__(self, input_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)


# ANN Classification Architecture
class ANNClassifier(nn.Module):
    def __init__(self, input_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )

    def forward(self, x):
        return self.network(x)


# Load Saved Artifacts
@st.cache_resource
def load_models():

    preprocessor = joblib.load(
        os.path.join(MODEL_PATH, "preprocessor.pkl")
    )

    thresholds = joblib.load(
        os.path.join(MODEL_PATH, "demand_thresholds.pkl")
    )

    input_features = len(preprocessor.get_feature_names_out())

    regressor = ANNRegressor(input_features).to(device)
    classifier = ANNClassifier(input_features).to(device)

    regressor.load_state_dict(
        torch.load(
            os.path.join(MODEL_PATH, "ann_regressor.pth"),
            map_location=device,
            weights_only=True
        )
    )

    classifier.load_state_dict(
        torch.load(
            os.path.join(MODEL_PATH, "ann_classifier.pth"),
            map_location=device,
            weights_only=True
        )
    )

    regressor.eval()
    classifier.eval()

    return preprocessor, thresholds, regressor, classifier


try:
    preprocessor, thresholds, regressor, classifier = load_models()

    st.sidebar.success("All models loaded successfully!")
    st.sidebar.write("Device:", device)
    st.sidebar.write("Input Features:", len(preprocessor.get_feature_names_out()))

except Exception as e:
    st.error(f"Model loading failed: {e}")
    st.stop()

Appending to /content/drive/MyDrive/SmartFood_Rescue/app.py


In [ ]:
app_path = "/content/drive/MyDrive/SmartFood_Rescue/app.py"

with open(app_path, "r") as file:
    app_code = file.read()

print("Total Lines:", len(app_code.splitlines()))
print("Model Loading Section:", "def load_models():" in app_code)
print("Regression Architecture:", "class ANNRegressor" in app_code)
print("Classification Architecture:", "class ANNClassifier" in app_code)

Total Lines: 133
Model Loading Section: True
Regression Architecture: True
Classification Architecture: True


In [ ]:
%%writefile -a /content/drive/MyDrive/SmartFood_Rescue/app.py

# Load Dataset for Input Options
@st.cache_data
def load_dataset():
    return pd.read_csv(
        os.path.join(DATASET_PATH, "train_prepared.csv")
    )

train_data = load_dataset()

# Application Input Section
st.markdown("---")
st.header("📋 Food Demand Prediction")

col1, col2 = st.columns(2)

with col1:
    st.subheader("Food Center Details")

    week = st.number_input(
        "Week",
        min_value=int(train_data["week"].min()),
        max_value=int(train_data["week"].max()),
        value=int(train_data["week"].min())
    )

    center_id = st.selectbox(
        "Fulfilment Center",
        sorted(train_data["center_id"].unique().tolist())
    )

    meal_id = st.selectbox(
        "Meal ID",
        sorted(train_data["meal_id"].unique().tolist())
    )

    category = st.selectbox(
        "Food Category",
        sorted(train_data["category"].unique().tolist())
    )

    cuisine = st.selectbox(
        "Cuisine",
        sorted(train_data["cuisine"].unique().tolist())
    )

with col2:
    st.subheader("Pricing and Promotion")

    checkout_price = st.number_input(
        "Checkout Price",
        min_value=0.01,
        value=100.0
    )

    base_price = st.number_input(
        "Base Price",
        min_value=0.01,
        value=120.0
    )

    emailer_for_promotion = st.selectbox(
        "Email Promotion",
        [0, 1],
        format_func=lambda x: "Yes" if x == 1 else "No"
    )

    homepage_featured = st.selectbox(
        "Homepage Featured",
        [0, 1],
        format_func=lambda x: "Yes" if x == 1 else "No"
    )

st.subheader("Prepared Food Quantity")

prepared_quantity = st.number_input(
    "Enter Prepared Quantity (Orders)",
    min_value=0,
    value=100,
    step=1
)

st.caption(
    "Enter the quantity of food you plan to prepare. "
    "This will be used later to estimate potential surplus."
)

Appending to /content/drive/MyDrive/SmartFood_Rescue/app.py


In [ ]:
%%writefile -a /content/drive/MyDrive/SmartFood_Rescue/app.py

st.markdown("---")
st.subheader("🏢 Additional Center Information")

col3, col4 = st.columns(2)

with col3:
    city_code = st.selectbox(
        "City Code",
        sorted(train_data["city_code"].unique().tolist())
    )

    region_code = st.selectbox(
        "Region Code",
        sorted(train_data["region_code"].unique().tolist())
    )

with col4:
    center_type = st.selectbox(
        "Center Type",
        sorted(train_data["center_type"].unique().tolist())
    )

    op_area = st.number_input(
        "Operating Area",
        min_value=0.01,
        value=float(train_data["op_area"].median())
    )

st.success("All required input fields are available!")

Appending to /content/drive/MyDrive/SmartFood_Rescue/app.py


In [ ]:
%%writefile -a /content/drive/MyDrive/SmartFood_Rescue/app.py

st.markdown("---")

# Prediction Button
if st.button("🔍 Predict Food Demand", use_container_width=True):

    # Feature Engineering
    discount_amount = base_price - checkout_price

    discount_percentage = (
        discount_amount / base_price
    ) * 100

    promotion_active = int(
        emailer_for_promotion == 1 or homepage_featured == 1
    )

    # Create Input DataFrame
    input_data = pd.DataFrame([{
        "week": week,
        "center_id": center_id,
        "meal_id": meal_id,
        "checkout_price": checkout_price,
        "base_price": base_price,
        "emailer_for_promotion": emailer_for_promotion,
        "homepage_featured": homepage_featured,
        "category": category,
        "cuisine": cuisine,
        "city_code": city_code,
        "region_code": region_code,
        "center_type": center_type,
        "op_area": op_area,
        "discount_amount": discount_amount,
        "discount_percentage": discount_percentage,
        "promotion_active": promotion_active
    }])

    # Maintain Original Feature Order
    input_data = input_data[feature_columns] if "feature_columns" in globals() else input_data

    st.write("Input Data Prepared Successfully!")
    st.dataframe(input_data)

Appending to /content/drive/MyDrive/SmartFood_Rescue/app.py


In [ ]:
%%writefile -a /content/drive/MyDrive/SmartFood_Rescue/app.py

    # Transform Input Using Saved Preprocessor
    processed_input = preprocessor.transform(input_data)

    if hasattr(processed_input, "toarray"):
        processed_input = processed_input.toarray()

    processed_input = np.asarray(
        processed_input,
        dtype=np.float32
    )

    input_tensor = torch.tensor(
        processed_input,
        dtype=torch.float32,
        device=device
    )

    # Model Predictions
    with torch.no_grad():

        # Regression Prediction
        regression_output = regressor(input_tensor)

        predicted_orders = float(
            np.expm1(regression_output.cpu().numpy()[0][0])
        )

        predicted_orders = max(0, predicted_orders)

        # Classification Prediction
        classification_output = classifier(input_tensor)

        predicted_class = int(
            torch.argmax(classification_output, dim=1).item()
        )

        predicted_demand = thresholds["class_mapping"][
            predicted_class
        ]

    # Display Results
    st.markdown("---")
    st.header("📊 Food Demand Prediction Results")

    result_col1, result_col2 = st.columns(2)

    with result_col1:
        st.metric(
            "Predicted Food Orders",
            f"{predicted_orders:.0f}"
        )

    with result_col2:
        st.metric(
            "Predicted Demand Level",
            predicted_demand
        )

    st.success("Both ANN models generated predictions successfully!")


Appending to /content/drive/MyDrive/SmartFood_Rescue/app.py


In [ ]:
%%writefile -a /content/drive/MyDrive/SmartFood_Rescue/app.py

    # Surplus Estimation
    estimated_surplus = max(
        0,
        prepared_quantity - predicted_orders
    )

    st.markdown("---")
    st.header("♻️ Food Surplus Estimation")

    surplus_col1, surplus_col2 = st.columns(2)

    with surplus_col1:
        st.metric(
            "Prepared Quantity",
            f"{prepared_quantity}"
        )

    with surplus_col2:
        st.metric(
            "Estimated Surplus",
            f"{estimated_surplus:.0f}"
        )

    if estimated_surplus > 0:
        st.warning(
            f"Approximately {estimated_surplus:.0f} orders "
            "may remain based on the demand forecast."
        )
    else:
        st.success(
            "No potential surplus is estimated based on the forecast."
        )

Appending to /content/drive/MyDrive/SmartFood_Rescue/app.py


In [ ]:
%%writefile -a /content/drive/MyDrive/SmartFood_Rescue/app.py

    # Food Rescue Recommendation Engine
    st.markdown("---")
    st.header("🌱 Smart Food Rescue Recommendations")

    if estimated_surplus <= 0:

        st.success(
            "Recommended Action: Normal Food Preparation"
        )

        st.write(
            "Your planned preparation quantity is within "
            "the predicted demand. No potential surplus "
            "has been estimated."
        )

    elif estimated_surplus <= prepared_quantity * 0.20:

        st.info(
            "Recommended Action: Redistribute Potential Surplus"
        )

        st.write(
            "Consider redistributing the estimated surplus "
            "to nearby food outlets or community organizations."
        )

    else:

        st.warning(
            "Recommended Action: Prioritize Food Rescue"
        )

        st.write(
            "A relatively large surplus is estimated. "
            "Consider arranging timely donation or redistribution "
            "through eligible food rescue organizations."
        )

    st.caption(
        "Food safety reminder: Only donate or redistribute food "
        "that has been safely prepared, stored, transported, "
        "and is suitable for consumption. Follow applicable "
        "food safety requirements."
    )

Appending to /content/drive/MyDrive/SmartFood_Rescue/app.py


In [ ]:
app_path = "/content/drive/MyDrive/SmartFood_Rescue/app.py"

with open(app_path, "r") as file:
    app_code = file.read()

checks = {
    "Streamlit Configuration": "st.set_page_config" in app_code,
    "Regression Model": "class ANNRegressor" in app_code,
    "Classification Model": "class ANNClassifier" in app_code,
    "Model Loading": "def load_models():" in app_code,
    "Input Interface": "Food Demand Prediction" in app_code,
    "Feature Engineering": "discount_percentage" in app_code,
    "Regression Prediction": "regressor(input_tensor)" in app_code,
    "Classification Prediction": "classifier(input_tensor)" in app_code,
    "Surplus Estimation": "estimated_surplus" in app_code,
    "Food Rescue Recommendations": "Smart Food Rescue Recommendations" in app_code
}

print("SMARTFOOD RESCUE APP VERIFICATION\n")

for section, status in checks.items():
    print(f"{section}: {'PASS' if status else 'FAIL'}")

print("\nTotal Lines:", len(app_code.splitlines()))

SMARTFOOD RESCUE APP VERIFICATION

Streamlit Configuration: PASS
Regression Model: PASS
Classification Model: PASS
Model Loading: PASS
Input Interface: PASS
Feature Engineering: PASS
Regression Prediction: PASS
Classification Prediction: PASS
Surplus Estimation: PASS
Food Rescue Recommendations: PASS

Total Lines: 431


In [ ]:
import ast

app_path = "/content/drive/MyDrive/SmartFood_Rescue/app.py"

with open(app_path, "r") as file:
    code = file.read()

try:
    ast.parse(code)
    print("Python Syntax: VALID")
    print("No syntax or indentation errors detected!")

except SyntaxError as e:
    print("Python Syntax: ERROR")
    print("Line:", e.lineno)
    print("Error:", e.msg)

Python Syntax: VALID
No syntax or indentation errors detected!
